# Context Engineering — Hands-On

**LLM Engineering · Domain 3 · Roadmap Week 15**

Offline notebook: rank, budget, order, and inspect toy context windows with stdlib + numpy only.

## 0. Setup: candidate chunks

In [ ]:
%pip install -q numpy
import numpy as np

def tokens(text): return len(text.split())

instructions = "Answer only from evidence. If missing, say insufficient context."
task = "What does the refund policy require?"
chunks = [
    {"id":"refund","text":"Refund policy requires a receipt ID and payment date before approval.","score":0.95,"source":"policy"},
    {"id":"shipping","text":"Shipping labels expire after thirty days.","score":0.20,"source":"policy"},
    {"id":"tax","text":"Tax corrections require finance review and a signed invoice.","score":0.45,"source":"policy"},
    {"id":"picnic","text":"The company picnic will include sandwiches and games.","score":0.05,"source":"wiki"},
]
print("candidate tokens", [(c["id"], tokens(c["text"])) for c in chunks])

## 1. Choose chunks by relevance density under a budget

In [ ]:
def assemble(candidates, budget):
    fixed = tokens(instructions) + tokens(task)
    remaining = budget - fixed
    ordered = sorted(candidates, key=lambda c: c["score"] / max(1, tokens(c["text"])), reverse=True)
    chosen = []
    for c in ordered:
        n = tokens(c["text"])
        if n <= remaining:
            chosen.append(c); remaining -= n
    return chosen, remaining

for budget in [22, 35, 55]:
    chosen, remaining = assemble(chunks, budget)
    print("budget", budget, "chosen", [c["id"] for c in chosen], "remaining", remaining)

## 2. Assemble with delimiters and traceability

In [ ]:
def render_context(chosen):
    evidence = "\n".join(f"<doc id='{c['id']}' source='{c['source']}'>{c['text']}</doc>" for c in chosen)
    ctx = f"SYSTEM: {instructions}\n\nEVIDENCE:\n{evidence}\n\nUSER: {task}"
    trace = [{"id":c["id"], "tokens":tokens(c["text"]), "score":c["score"]} for c in chosen]
    return ctx, trace

chosen, _ = assemble(chunks, 35)
ctx, trace = render_context(chosen)
print(ctx)
print("TRACE", trace)

## 3. Lost-in-the-middle toy probe

In [ ]:
def edge_weight(position, n):
    x = position / max(1, n - 1)
    return 0.5 + abs(x - 0.5)  # edges higher than middle

def retrieval_signal(sequence, keyword):
    vals = []
    for i, text in enumerate(sequence):
        if keyword in text.lower(): vals.append(edge_weight(i, len(sequence)))
    return max(vals) if vals else 0.0

middle = ["instructions", "filler A", "refund policy requires receipt", "filler B", "question"]
edge = ["instructions", "refund policy requires receipt", "filler A", "filler B", "question"]
print("middle signal", retrieval_signal(middle, "refund"))
print("edge signal  ", retrieval_signal(edge, "refund"))

## 4. Compression: extractive vs abstractive

In [ ]:
doc = "Refund policy requires receipt ID, payment date, and manager approval for amounts over 500 dollars."
extractive = "Refund policy requires receipt ID, payment date, and manager approval for amounts over 500 dollars."
abstractive = "Refunds need documentation and sometimes approval."
for name, text in [("extractive", extractive), ("abstractive", abstractive)]:
    has_amount = "500" in text
    print(name, "tokens", tokens(text), "preserves threshold?", has_amount)

## 5. Memory selection is a query, not a dump

In [ ]:
memories = [
    {"id":"m1","text":"User prefers concise bullet answers.","tags":{"style"}},
    {"id":"m2","text":"User once asked about Kubernetes refunds metaphor.","tags":{"old","irrelevant"}},
    {"id":"m3","text":"User is working on support automation.","tags":{"project"}},
]
def select_memory(memories, allowed_tags):
    return [m for m in memories if m["tags"] & allowed_tags]
print("selected", [m["id"] for m in select_memory(memories, {"style","project"})])
print("not dumped", [m["id"] for m in memories if m not in select_memory(memories, {"style","project"})])

## 6. Exercises
1. Add recency as a second ranking signal.
2. Force the current task to appear last and critical evidence second.
3. Add conflict resolution: source-of-record retrieval beats memory.
4. Log an error report explaining which chunk caused an incorrect answer.

## Links
- Literature note: `02 Literature Notes/LLM Engineering/Context Engineering`
- Snippets: `04 Code Snippets/LLM/Context Budget Assembler`, `.../Lost In The Middle Probe`
- MOC: `06 Maps of Content/LLM Engineering Concepts`